
# Imports and Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, expr, col, when, lit, regexp_extract, percentile_approx, regexp_replace, expr, count, to_timestamp, unix_timestamp, mean
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
import pyspark.sql.types as T
from pyspark.sql.window import Window

from pyspark.sql.types import *
from pyspark.ml.classification import LogisticRegression

from pyspark.storagelevel import StorageLevel
from graphframes import GraphFrame

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import holidays

Goal: Need to read the data, select the needed columns, cast and clean columns, fill in nulls, conduct l1/l2 regularization check, split into train/val/test, undersample the train data, normalize and vectorize the datasets, save to results clean the notebook

In [0]:
# Sanity variables

verbose = True

In [0]:
CUSTOM_JOIN_BASE = "/tmp/sunil-thakur/custom_join/flight_weather_features/all"

display(dbutils.fs.ls(f"{CUSTOM_JOIN_BASE}"))

# Checkpointing Info
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

In [0]:
df_base = spark.read.parquet(f"{CUSTOM_JOIN_BASE}")

In [0]:
if verbose:
    display(df_base, limit = 10)

In [0]:
if verbose:
    print(len(df_base.columns))

In [0]:
if verbose:
    df_base.printSchema()

# Null Checks

In [0]:
df_impute = df_base

In [0]:
neighbor_cols = [c for c in df_impute.columns if c.startswith("neighbor_") or c.endswith("_neighbor")]
fallback_cols = [c for c in df_impute.columns if c.endswith("_fallback")]
if verbose:
    print("neighbor_cols:", len(neighbor_cols))
    print(neighbor_cols)
    print("fallback_cols:", len(fallback_cols))
    print(fallback_cols)

In [0]:
df_impute = df_impute.drop("DEP_DELAY")

In [0]:
# remove "DEP_DELAY"
# want to drop neighbor and fallback cols
df_impute = df_impute.drop(*neighbor_cols, *fallback_cols)
if verbose:
    print(len(df_impute.columns))
    df_impute.printSchema()

In [0]:
if verbose:
    # Dataset shape
    total = df_impute.count()
    ncols = len(df_impute.columns)
    print(f"Records: {total:,}  |  Columns: {ncols}")

    # All columns except neighbor_ and _fallback
    check_cols = [c for c in df_impute.columns
                if not c.startswith("neighbor_") and not c.endswith("_fallback") and not c.endswith("_neighbor")]

    # Compute null %
    null_exprs = [
        (100 * F.count(F.when(F.col(c).isNull(), 1)) / total).alias(c)
        for c in check_cols
    ]
    missing = (
        df_impute.select(null_exprs)
        .toPandas().T.reset_index()
        .rename(columns={"index": "column", 0: "null_pct"})
        .sort_values("null_pct", ascending=False)
    )
    print(f"\nNull % for {len(check_cols)} columns (excluding neighbor & fallback):")
    display(missing)


# Imputations

In [0]:
if verbose:
    df_impute.printSchema()

In [0]:
# Impute Nulls
# Phase 1: safe fills before splitting (domain rules only, no leakage risk)

before = df_impute.count()

# drop cancelled flights (no target) and unmatched airports (no coords)
df_impute = df_impute.filter(F.col("DEP_DEL15").isNotNull())
print(f"Dropped {before - df_impute.count():,} cancelled flights (null DEP_DEL15)")

before_coords = df_impute.count()
df_impute = df_impute.filter(F.col("dest_lat").isNotNull() & F.col("dest_lon").isNotNull())
print(f"Dropped {before_coords - df_impute.count():,} rows with missing dest coords")

# trend/deviation cols: null means no history, so deviation is 0
df_impute = df_impute.fillna(0, subset=[
    "trend_7d", "volatility_7d",
    "delta_7d_vs_hist", "delta_15d_vs_hist", "delta_30d_vs_hist",
    "delta_morning_vs_hist",
])

# ratio null means no morning data yet, 1.0 = neutral ("same as historical")
df_impute = df_impute.fillna(1.0, subset=["ratio_morning_vs_hist"])

df_impute.unpersist()
df_impute = df_impute.cache()
phase1_count = df_impute.count()
print(f"\nPhase 1 done: {phase1_count:,} rows cached")

# check what still has nulls heading into Phase 2
null_counts = df_impute.select([
    F.count(F.when(F.col(col).isNull(), 1)).alias(col)
    for col in df_impute.columns
]).toPandas().T.reset_index().rename(columns={"index": "column", 0: "null_count"})

null_counts = null_counts[null_counts["null_count"] > 0].sort_values("null_count", ascending=False)
null_counts["null_pct"] = round(100 * null_counts["null_count"] / phase1_count, 2)

print(f"{len(null_counts)} columns still have nulls (Phase 2 handles these after the split):")
display(null_counts)

In [0]:
print(df_impute.columns)

In [0]:
numeric_cols = [c for c in df_impute.columns if c != "DEP_DEL15" and c in dict(df_impute.dtypes) and dict(df_impute.dtypes)[c] != 'string']
numeric_cols

In [0]:
df_numeric = df_impute.select(numeric_cols + ["DEP_DEL15"]).cache()
if verbose:
    df_numeric.printSchema()

In [0]:
# Cast DEP_DEL15 to integer
df_numeric = df_numeric.withColumn("DEP_DEL15", F.col('DEP_DEL15').cast("integer"))


# L1/L2 Regularization Check

In [0]:
if verbose:
    print(numeric_cols)

In [0]:
df_numeric.printSchema()

In [0]:
dtype_map = dict(df_numeric.dtypes)
feature_names = [c for c in numeric_cols if c != "DEP_DEL15" and c!= "DEP_DELAY" and dtype_map.get(c) in ('double', 'int', 'float', 'bigint', 'long')]

In [0]:
if verbose:
    print(feature_names)

In [0]:
numeric_cols = feature_names + ["DEP_DEL15"]

In [0]:
# L1/L2 Regularization Check's Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_numeric.select(*(numeric_cols)).dropna()
assembler = VectorAssembler(inputCols=feature_names, outputCol="features")
df_vec_f = assembler.transform(df_corr).withColumn("label", F.col("DEP_DEL15").cast("double"))
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_f = scaler.fit(df_vec_f)
df_model = scaler_model_f.transform(df_vec_f) #.select("scaled_features")

df_model = df_model.select("scaled_features", "label")
df_model = df_model.withColumnRenamed("scaled_features", "features")

In [0]:
lr_l1 = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    regParam=0.01, 
    elasticNetParam=1.0   # 1.0 = L1 (Lasso)
)

model_l1 = lr_l1.fit(df_model)

In [0]:
lr_l2 = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    regParam=0.01,
    elasticNetParam=0.0   # L2 regularization
)

model_l2 = lr_l2.fit(df_model)

In [0]:
coefficients_l1 = model_l1.coefficients.toArray()

In [0]:
coefficients_l2 = model_l2.coefficients.toArray()

In [0]:
if verbose:
    display(coefficients_l1)

In [0]:
if verbose:
    display(coefficients_l2)

In [0]:
l1_all_feature_coeff_pairs = list(zip(feature_names, coefficients_l1))
l1_coef_df = pd.DataFrame(l1_all_feature_coeff_pairs, columns=["feature", "coefficient"])
l1_coef_df["abs_coefficient"] = l1_coef_df["coefficient"].abs()
l1_coef_df = l1_coef_df.sort_values("abs_coefficient", ascending=False)

In [0]:
visual_l1_coef_df = l1_coef_df.sort_values("abs_coefficient", ascending=False)[l1_coef_df["coefficient"] != 0.0]
if verbose:
    display(l1_coef_df)
    display(visual_l1_coef_df)

In [0]:
l2_all_feature_coeff_pairs = list(zip(feature_names, coefficients_l2))
l2_coef_df = pd.DataFrame(l2_all_feature_coeff_pairs, columns=["feature", "coefficient"])
l2_coef_df["abs_coefficient"] = l2_coef_df["coefficient"].abs()
l2_coef_df = l2_coef_df.sort_values("abs_coefficient", ascending=False)

In [0]:
visual_l2_coef_df = l2_coef_df.sort_values("abs_coefficient", ascending=False)[:20]
if verbose:
    display(l2_coef_df)
    display(visual_l2_coef_df)

In [0]:
if verbose:
    plt.figure(figsize=(10, 12))
    plt.barh(visual_l1_coef_df["feature"], visual_l1_coef_df["coefficient"])
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.title("Logistic Regression with L1 regularization Feature Coefficients")
    plt.axvline(x=0, color='r', linestyle='--')
    plt.gca().invert_yaxis()  # largest at top
    plt.tight_layout()

    plt.show()

In [0]:
if verbose:
    plt.figure(figsize=(10, 12))
    plt.barh(visual_l2_coef_df["feature"], visual_l2_coef_df["coefficient"])
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.title("Logistic Regression with L2 regularization Feature Coefficients")
    plt.axvline(x=0, color='r', linestyle='--')
    plt.gca().invert_yaxis()  # largest at top
    plt.tight_layout()
    plt.show()

In [0]:
all_coef_df = l1_coef_df.merge(l2_coef_df, how="inner", on="feature", suffixes=("_l1", "_l2"))

In [0]:
display(all_coef_df)

In [0]:
# want to get the deduplicated list of features from visual_l2_coef_df plus visual_l1_coef_df
best_regularization_features = list(set(visual_l1_coef_df["feature"]) | set(visual_l2_coef_df["feature"]))

In [0]:
print(all_coef_df)

In [0]:
spark.createDataFrame(all_coef_df) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv(f"{folder_path}/custom-join/custom_results/regularization_coefficients_base_dataset.csv")

# Final Feature Selection

In [0]:
best_regularization_features

In [0]:
### MAKE SURE TO GET RID OF DEP_DELAY
targetVariable = ['DEP_DEL15']

partitionColumns = ["YEAR", "MONTH"]

# finalTemporalColsSelected = [ 
#     'tail_avg_delay_10h',
#     'tail_recent_delay',
#     'route_avg_delay_6h',
#     'route_max_delay_6h',
#     'dep_delay_1h',
#     'dep_delay_3h',
#     'dep_delay_6h',
#     'num_flights_3h'
# ]

# finalTimeColsSelected = [
#     'CRS_DEP_TIME',
#     # 'CRS_ARR_TIME', # not present in Custom Join
#     'YEAR',
#     'MONTH',
#     'QUARTER',
#     'DAY_OF_MONTH',
#     'DAY_OF_WEEK'
#  ]

# finalWeatherColsSelected = [
#     "HourlyPrecipitation",
#     "HourlyWindGustSpeed",
#     "HourlyWindSpeed",
#     "HourlyDryBulbTemperature",
#     "HourlyDewPointTemperature",
#     "HourlyRelativeHumidity",
#     "HourlyVisibility",
#     "has_rain",
#     "has_snow",
#     "has_fog",
#     "has_thunderstorm",
#     "has_freezing",
#     "has_haze",
#     "sky_cover_ordinal",     # worst cloud cover in window
#     "has_ceiling",          # any ceiling restriction in window
#     "ceiling_ft"
# ]

# finalFlightsColsSelected = [
#     'DISTANCE',
#     'days_to_nearest_holiday',
#     'dep_volume_window',
#     'arr_volume_window',
# ]

# finalGraphColsSelected = ['num_flights_3h',
#     'deg_in',
#     'deg_out',
#     'graph_pagerank',
#     'hub_score',
#     'authority_score',
#     'graph_betweenness'
# ]

# additionalColsSelected = [
#     "dep_delay_1h", "dep_delay_3h", "dep_delay_6h", "max_dep_delay_3h",
#     "lag_1d_delay_rate", "lag_7d_delay_rate", "lag_15d_delay_rate", "lag_30d_delay_rate",
#     "morning_delay_rate_airport_day", "avg_morning_delay_rate_airport",
#     "route_avg_delay_6h", "route_max_delay_6h",
# ]

In [0]:
# selectedCols = list(set(targetVariable + finalTemporalColsSelected + finalTimeColsSelected + finalWeatherColsSelected + finalFlightsColsSelected + finalGraphColsSelected + additionalColsSelected))

selectedCols = list(targetVariable + partitionColumns + best_regularization_features)
df_features = df_numeric.select(selectedCols)

In [0]:
if verbose:
    df_features.printSchema()
    display(df_features, limit = 10)

In [0]:
# Quick checkpoint of data
#remove df_features_basegraph_dataset.parquet
dbutils.fs.rm(f"{folder_path}/custom-join/all/checkpoint/df_features_basegraph_dataset.parquet", True)
df_features.write.mode("overwrite").parquet("{file_path}/custom-join/all/checkpoint/df_features_basegraph_dataset.parquet")


# Data Split

In [0]:
df_train = df_features.filter(df_features["YEAR"].isin([2015, 2016, 2017, 2018]))
df_test = df_features.filter(df_features["YEAR"] == 2019)


# Undersampling Data

In [0]:

# Assuming 'df' and binary column 'label' (1=minority, 0=majority)
major_df = df_train.filter(col("DEP_DEL15") == 0)
minor_df = df_train.filter(col("DEP_DEL15") == 1)

# Calculate ratio
ratio = float(minor_df.count()) / float(major_df.count())
print("ratio = ", ratio)

# Sample majority
sampled_major_df = major_df.sample(withReplacement=False, fraction=ratio, seed=42)

# Combine datasets
df_train = sampled_major_df.union(minor_df)

# Check results
df_train.groupBy("DEP_DEL15").count().show()


# Normalization and Vectorization of Data

In [0]:
selectedCols

In [0]:
print(numeric_cols)

In [0]:
# Normalization - Train
feature_cols = [c for c in selectedCols if c not in ("DEP_DEL15", "YEAR", "MONTH")]
df_corr_train = df_train.select(*selectedCols).dropna()

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec_train = assembler.transform(df_corr_train).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_train = scaler.fit(df_vec_train)
df_scaled_custom_train = scaler_model_train.transform(df_vec_train)

df_scaled_custom_train = df_scaled_custom_train.select("YEAR", "MONTH", "scaled_features", "label")
df_scaled_custom_train = df_scaled_custom_train.withColumnRenamed("scaled_features", "features")

In [0]:
# Normalization - Test
feature_cols = [c for c in selectedCols if c not in ("DEP_DEL15", "YEAR", "MONTH")]
df_corr_test = df_test.select(*selectedCols).dropna()

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_vec_test = assembler.transform(df_corr_test).withColumn("label", col("DEP_DEL15").cast("double"))
df_scaled_custom_test = scaler_model_train.transform(df_vec_test)

df_scaled_custom_test = df_scaled_custom_test.select("YEAR", "MONTH", "scaled_features", "label")
df_scaled_custom_test = df_scaled_custom_test.withColumnRenamed("scaled_features", "features")

# Save to File System

In [0]:
#2015-2018
df_scaled_custom_train.write.mode("overwrite").partitionBy("YEAR", "MONTH").parquet(f"{folder_path}/custom-join/all/data_separated/train")
print("Completed train_val: Path: ", f"{folder_path}/custom-join/all/data_separated/train")

#2019
df_scaled_custom_test.write.mode("overwrite").partitionBy("YEAR", "MONTH").parquet(f"{folder_path}/custom-join/all/data_separated/test")
print("Completed test: Path: ", f"{folder_path}/custom-join/all/data_separated/test")
